# Multi-Domain Dynamic Systems Modeling, Simulation and Control

This notebook connects subsystems in the same spirit as a Simulink block diagram.

### Included integrated examples
- electrical controller + DC motor + mechanical load
- controller + sensor + actuator + plant + feedback
- electro-pneumatic positioning
- electro-hydraulic positioning
- cascade control
- feedforward control
- disturbance/noise/saturation in connected systems

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import scipy.linalg as la
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

try:
    import control as ct
    HAS_CONTROL = True
except Exception as e:
    HAS_CONTROL = False
    print("Install python-control with: pip install control")
    print(e)

In [2]:
def stability_from_poles(poles, tol=1e-8):
    p=np.asarray(poles,dtype=complex)
    if np.any(np.real(p)>tol): return "UNSTABLE"
    if np.all(np.real(p)<-tol): return "ASYMPTOTICALLY STABLE"
    return "MARGINALLY STABLE"

def damping_label(zeta,tol=.02):
    if zeta < 0: return "UNSTABLE / NEGATIVE DAMPING"
    if abs(zeta)<1e-8: return "UNDAMPED"
    if zeta < 1-tol: return "UNDERDAMPED"
    if zeta <= 1+tol: return "CRITICALLY DAMPED (approximately)"
    return "OVERDAMPED"

def modal_label(poles):
    p=np.asarray(poles,dtype=complex)
    st=stability_from_poles(p)
    cp=[z for z in p if abs(np.imag(z))>1e-8]
    if cp:
        d=max(cp,key=lambda z:np.real(z))
        zeta=-np.real(d)/abs(d)
        if st=="UNSTABLE": return "OSCILLATORY UNSTABLE",st,zeta
        if zeta<.98: return "UNDERDAMPED",st,zeta
        if zeta<=1.02: return "CRITICALLY DAMPED-LIKE",st,zeta
        return "OVERDAMPED-LIKE",st,zeta
    return ("OVERDAMPED-LIKE / NON-OSCILLATORY" if st=="ASYMPTOTICALLY STABLE" else st),st,None

def metrics(t,y,final=None):
    t=np.asarray(t); y=np.asarray(y)
    if final is None: final=y[-1]
    peak_i=int(np.argmax(y)); peak=y[peak_i]
    overshoot=np.nan if abs(final)<1e-12 else max(0,(peak-final)/abs(final)*100)
    def cross(level):
        idx=np.where(y>=level)[0] if final>=0 else np.where(y<=level)[0]
        return t[idx[0]] if len(idx) else np.nan
    t10,t90=cross(.1*final),cross(.9*final)
    rise=t90-t10 if np.isfinite(t10) and np.isfinite(t90) else np.nan
    band=.02*max(abs(final),1e-12)
    out=np.where(np.abs(y-final)>band)[0]
    settle=0 if len(out)==0 else (t[out[-1]+1] if out[-1]<len(t)-1 else np.nan)
    return dict(rise_time=rise,settling_time=settle,peak_time=t[peak_i],overshoot_percent=overshoot,peak_value=peak)

def routh_table(coeffs):
    a=np.asarray(coeffs,float)
    n=len(a)-1; c=int(np.ceil((n+1)/2))
    R=np.zeros((n+1,c))
    R[0,:len(a[0::2])]=a[0::2]
    R[1,:len(a[1::2])]=a[1::2]
    eps=1e-9
    for i in range(2,n+1):
        if abs(R[i-1,0])<eps: R[i-1,0]=eps
        for j in range(c-1):
            R[i,j]=(R[i-1,0]*R[i-2,j+1]-R[i-2,0]*R[i-1,j+1])/R[i-1,0]
    s=np.sign(R[:,0]); changes=int(np.sum(s[:-1]*s[1:]<0))
    return R,changes

def jury_second_order(a):
    a0,a1,a2=map(float,a)
    cond={"|a2|<a0":abs(a2)<a0,"P(1)>0":a0+a1+a2>0,"P(-1)>0":a0-a1+a2>0}
    return cond,all(cond.values())

## 1. DC Motor Driving a Mechanical Load — Fully Interactive

This section combines an electrical motor model and a mechanical load.

Adjustable inputs:

- Supply voltage \(V\)
- Armature resistance \(R\)
- Armature inductance \(L\)
- Back-emf constant \(K_e\)
- Torque constant \(K_t\)
- Motor inertia \(J_m\)
- Load inertia \(J_L\)
- Mechanical damping \(B\)
- Load stiffness \(K_L\)
- Simulation time

Electrical equation:

\[
L\dot i + Ri + K_e\omega = V
\]

Mechanical equation:

\[
(J_m+J_L)\dot\omega + B\omega + K_L\theta = K_t i
\]

In [ ]:
def motor_load_interactive(
    V=24.0,
    R=2.0,
    L=0.5,
    Ke=0.1,
    Kt=0.1,
    Jmotor=0.02,
    Jload=0.05,
    B=0.04,
    Kload=1.0,
    sim_time=5.0
):
    J = Jmotor + Jload

    A = np.array([
        [-R/L, -Ke/L, 0],
        [Kt/J, -B/J, -Kload/J],
        [0, 1, 0]
    ])

    poles = np.linalg.eigvals(A)
    print("="*65)
    print("ELECTROMECHANICAL OPEN-LOOP ANALYSIS")
    print("="*65)
    print("Poles:", poles)
    print("Stability:", stability_from_poles(poles))
    print("="*65)

    def model(t, y):
        current, omega, theta = y
        di = (V - R*current - Ke*omega) / L
        domega = (Kt*current - B*omega - Kload*theta) / J
        dtheta = omega
        return [di, domega, dtheta]

    t = np.linspace(0, sim_time, 1800)
    sol = solve_ivp(model, [0, sim_time], [0, 0, 0], t_eval=t)

    current, omega, theta = sol.y

    plt.figure(figsize=(10,4))
    plt.plot(t, theta)
    plt.xlabel("Time (s)")
    plt.ylabel("Angular position (rad)")
    plt.title("Motor + Mechanical Load Position")
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(t, omega)
    plt.xlabel("Time (s)")
    plt.ylabel("Angular velocity (rad/s)")
    plt.title("Motor / Load Speed")
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(t, current)
    plt.xlabel("Time (s)")
    plt.ylabel("Current (A)")
    plt.title("Armature Current")
    plt.grid(True)
    plt.show()

widgets.interact(
    motor_load_interactive,
    V=widgets.FloatSlider(value=24,min=1,max=60,step=1,description="Voltage"),
    R=widgets.FloatSlider(value=2,min=.1,max=10,step=.1,description="R ohm"),
    L=widgets.FloatSlider(value=.5,min=.01,max=2,step=.01,description="L H"),
    Ke=widgets.FloatSlider(value=.1,min=.01,max=1,step=.01,description="Ke"),
    Kt=widgets.FloatSlider(value=.1,min=.01,max=1,step=.01,description="Kt"),
    Jmotor=widgets.FloatSlider(value=.02,min=.001,max=.2,step=.001,description="J motor"),
    Jload=widgets.FloatSlider(value=.05,min=0,max=.5,step=.01,description="J load"),
    B=widgets.FloatSlider(value=.04,min=.001,max=.5,step=.005,description="Damping"),
    Kload=widgets.FloatSlider(value=1,min=0,max=10,step=.25,description="Load K"),
    sim_time=widgets.FloatSlider(value=5,min=1,max=15,step=.5,description="Time")
);

interactive(children=(FloatSlider(value=24.0, description='Voltage', max=60.0, min=1.0, step=1.0), FloatSlider…

## 2. Closed-Loop Electromechanical Position Control — Fully Interactive

The controller is

\[
V_c=K_pe+K_i\int e\,dt-K_d\omega
\]

This section also allows the user to vary:

- PID gains
- Reference position
- Voltage saturation
- Sensor-noise amplitude
- Load-disturbance torque
- Disturbance start time

In [ ]:
def motor_pid_interactive(
    Kp=20.0,
    Ki=5.0,
    Kd=2.0,
    Vlim=24.0,
    ref=1.0,
    noise_amp=0.005,
    disturbance_torque=0.05,
    disturbance_time=3.0,
    R=2.0,
    L=0.5,
    Ke=0.1,
    Kt=0.1,
    J=0.07,
    B=0.04,
    Kload=1.0,
    sim_time=8.0
):
    def model(t, y):
        current, omega, theta, integral = y

        noise = noise_amp*np.sin(40*t)
        measured = theta + noise
        error = ref - measured

        voltage_cmd = Kp*error + Ki*integral - Kd*omega
        voltage = np.clip(voltage_cmd, -Vlim, Vlim)

        disturbance = disturbance_torque if t >= disturbance_time else 0.0

        di = (voltage - R*current - Ke*omega) / L
        domega = (Kt*current - B*omega - Kload*theta - disturbance) / J
        dtheta = omega
        dintegral = error

        return [di, domega, dtheta, dintegral]

    t = np.linspace(0, sim_time, 2200)
    sol = solve_ivp(model, [0, sim_time], [0,0,0,0], t_eval=t)

    theta = sol.y[2]

    plt.figure(figsize=(10,4))
    plt.plot(t, theta, label="Position")
    plt.axhline(ref, linestyle="--", label="Reference")
    plt.axvline(disturbance_time, linestyle=":", label="Disturbance starts")
    plt.xlabel("Time (s)")
    plt.ylabel("Position (rad)")
    plt.title("Closed-Loop Electromechanical Position Control")
    plt.grid(True)
    plt.legend()
    plt.show()

widgets.interact(
    motor_pid_interactive,
    Kp=widgets.FloatSlider(value=20,min=0,max=100,step=2,description="Kp"),
    Ki=widgets.FloatSlider(value=5,min=0,max=50,step=1,description="Ki"),
    Kd=widgets.FloatSlider(value=2,min=0,max=20,step=.5,description="Kd"),
    Vlim=widgets.FloatSlider(value=24,min=5,max=60,step=1,description="V limit"),
    ref=widgets.FloatSlider(value=1,min=.1,max=3,step=.1,description="Reference"),
    noise_amp=widgets.FloatSlider(value=.005,min=0,max=.05,step=.001,description="Noise"),
    disturbance_torque=widgets.FloatSlider(value=.05,min=0,max=.5,step=.01,description="Disturb."),
    disturbance_time=widgets.FloatSlider(value=3,min=.5,max=7,step=.5,description="Dist time"),
    R=widgets.FloatSlider(value=2,min=.1,max=10,step=.1,description="R"),
    L=widgets.FloatSlider(value=.5,min=.01,max=2,step=.01,description="L"),
    Ke=widgets.FloatSlider(value=.1,min=.01,max=1,step=.01,description="Ke"),
    Kt=widgets.FloatSlider(value=.1,min=.01,max=1,step=.01,description="Kt"),
    J=widgets.FloatSlider(value=.07,min=.005,max=.5,step=.005,description="J"),
    B=widgets.FloatSlider(value=.04,min=.001,max=.5,step=.005,description="B"),
    Kload=widgets.FloatSlider(value=1,min=0,max=10,step=.25,description="K load"),
    sim_time=widgets.FloatSlider(value=8,min=2,max=15,step=.5,description="Time")
);

interactive(children=(FloatSlider(value=20.0, description='Kp', step=2.0), FloatSlider(value=5.0, description=…

## 3. Electro-Pneumatic Positioning — Fully Interactive

You can vary both control parameters and pneumatic/mechanical plant parameters:

- \(K_p, K_i\)
- reference position
- piston area
- mass
- damping
- stiffness
- pressure time constant
- spool time constant
- pressure/velocity coupling
- maximum pressure
- external load
- sensor noise

In [5]:
def electro_pneu_interactive(
    Kp=4.0,
    Ki=1.0,
    ref=.25,
    piston_area=30e-4,
    mass=20.0,
    damping=120.0,
    stiffness=250.0,
    pressure_tau=.15,
    spool_tau=.03,
    pressure_velocity_coupling=8e5,
    pressure_limit=6e5,
    load=300.0,
    noise_amp=.001,
    sim_time=6.0
):
    def model(t, y):
        x, v, dp, spool, integral = y

        measured = x + noise_amp*np.sin(25*t)
        error = ref - measured

        valve_command = np.clip(Kp*error + Ki*integral, -1, 1)
        dspool = (valve_command - spool) / spool_tau

        pressure_command = spool * pressure_limit

        dx = v
        dv = (piston_area*dp - damping*v - stiffness*x - load) / mass
        ddp = (pressure_command - dp) / pressure_tau - pressure_velocity_coupling*v
        dintegral = error

        return [dx,dv,ddp,dspool,dintegral]

    t = np.linspace(0, sim_time, 2000)
    sol = solve_ivp(model,[0,sim_time],[0,0,0,0,0],t_eval=t)

    position = sol.y[0]
    pressure = sol.y[2]
    spool = sol.y[3]

    plt.figure(figsize=(10,4))
    plt.plot(t,position,label="Position")
    plt.axhline(ref,linestyle="--",label="Reference")
    plt.grid(True)
    plt.legend()
    plt.title("Electro-Pneumatic Position")
    plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(t,pressure/1e5)
    plt.ylabel("Pressure difference (bar)")
    plt.grid(True)
    plt.title("Pneumatic Pressure Difference")
    plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(t,spool)
    plt.ylabel("Normalized spool position")
    plt.grid(True)
    plt.title("Valve Spool Position")
    plt.show()

widgets.interact(
    electro_pneu_interactive,
    Kp=widgets.FloatSlider(value=4,min=0,max=20,step=.5,description="Kp"),
    Ki=widgets.FloatSlider(value=1,min=0,max=10,step=.2,description="Ki"),
    ref=widgets.FloatSlider(value=.25,min=.05,max=.5,step=.05,description="Reference"),
    piston_area=widgets.FloatSlider(value=30e-4,min=5e-4,max=100e-4,step=5e-4,readout_format=".4f",description="Area m²"),
    mass=widgets.FloatSlider(value=20,min=1,max=100,step=1,description="Mass"),
    damping=widgets.FloatSlider(value=120,min=0,max=800,step=10,description="Damping"),
    stiffness=widgets.FloatSlider(value=250,min=20,max=1500,step=10,description="Stiffness"),
    pressure_tau=widgets.FloatSlider(value=.15,min=.02,max=1,step=.01,description="P tau"),
    spool_tau=widgets.FloatSlider(value=.03,min=.005,max=.2,step=.005,description="Spool tau"),
    pressure_velocity_coupling=widgets.FloatSlider(value=8e5,min=0,max=3e6,step=1e5,description="Coupling"),
    pressure_limit=widgets.FloatSlider(value=6e5,min=1e5,max=12e5,step=.5e5,description="P max"),
    load=widgets.FloatSlider(value=300,min=0,max=3000,step=50,description="Load"),
    noise_amp=widgets.FloatSlider(value=.001,min=0,max=.02,step=.001,description="Noise"),
    sim_time=widgets.FloatSlider(value=6,min=2,max=15,step=.5,description="Time")
);

interactive(children=(FloatSlider(value=4.0, description='Kp', max=20.0, step=0.5), FloatSlider(value=1.0, des…

## 4. Electro-Hydraulic Positioning — Fully Interactive

Adjustable hydraulic and control parameters include:

- controller gains
- reference position
- piston area
- moving mass
- damping and stiffness
- bulk modulus
- effective fluid volume
- leakage coefficient
- servo-valve time constant
- maximum flow
- external load
- sensor noise

In [ ]:
def electro_hyd_interactive(
    Kp=5e-4,
    Ki=1e-4,
    ref=.15,
    piston_area=50e-4,
    mass=100.0,
    damping=1200.0,
    stiffness=5000.0,
    beta=1.5e9,
    fluid_volume=.01,
    leakage=2e-12,
    spool_tau=.02,
    max_flow_lpm=60.0,
    load=5000.0,
    noise_amp=.0005,
    sim_time=3.0
):
    max_flow = max_flow_lpm/60000

    def model(t, y):
        x,v,dp,spool,integral = y

        measured = x + noise_amp*np.sin(30*t)
        error = ref - measured

        valve_cmd = np.clip(Kp*error + Ki*integral, -1, 1)
        dspool = (valve_cmd-spool)/spool_tau

        flow = spool*max_flow

        dx = v
        dv = (piston_area*dp - damping*v - stiffness*x - load)/mass
        ddp = (beta/fluid_volume)*(flow - 2*piston_area*v - leakage*dp)
        dintegral = error

        return [dx,dv,ddp,dspool,dintegral]

    t=np.linspace(0,sim_time,2200)
    sol=solve_ivp(model,[0,sim_time],[0,0,0,0,0],t_eval=t,method="Radau")

    position=sol.y[0]
    pressure=sol.y[2]
    spool=sol.y[3]

    plt.figure(figsize=(10,4))
    plt.plot(t,position,label="Position")
    plt.axhline(ref,linestyle="--",label="Reference")
    plt.grid(True)
    plt.legend()
    plt.title("Electro-Hydraulic Position")
    plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(t,pressure/1e6)
    plt.ylabel("Pressure difference (MPa)")
    plt.grid(True)
    plt.title("Hydraulic Pressure Difference")
    plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(t,spool)
    plt.ylabel("Normalized spool position")
    plt.grid(True)
    plt.title("Servo-Valve Position")
    plt.show()

widgets.interact(
    electro_hyd_interactive,
    Kp=widgets.FloatLogSlider(value=5e-4,base=10,min=-5,max=-2,step=.1,description="Kp"),
    Ki=widgets.FloatLogSlider(value=1e-4,base=10,min=-6,max=-3,step=.1,description="Ki"),
    ref=widgets.FloatSlider(value=.15,min=.02,max=.4,step=.02,description="Reference"),
    piston_area=widgets.FloatSlider(value=50e-4,min=10e-4,max=200e-4,step=5e-4,readout_format=".4f",description="Area m²"),
    mass=widgets.FloatSlider(value=100,min=10,max=1000,step=10,description="Mass"),
    damping=widgets.FloatSlider(value=1200,min=0,max=10000,step=100,description="Damping"),
    stiffness=widgets.FloatSlider(value=5000,min=100,max=30000,step=500,description="Stiffness"),
    beta=widgets.FloatSlider(value=1.5e9,min=.3e9,max=2.5e9,step=.1e9,description="Bulk Pa"),
    fluid_volume=widgets.FloatSlider(value=.01,min=.001,max=.05,step=.001,description="Volume m³"),
    leakage=widgets.FloatLogSlider(value=2e-12,base=10,min=-13,max=-9,step=.1,description="Leakage"),
    spool_tau=widgets.FloatSlider(value=.02,min=.005,max=.1,step=.005,description="Spool tau"),
    max_flow_lpm=widgets.FloatSlider(value=60,min=5,max=150,step=5,description="Flow L/min"),
    load=widgets.FloatSlider(value=5000,min=0,max=50000,step=500,description="Load"),
    noise_amp=widgets.FloatSlider(value=.0005,min=0,max=.01,step=.0005,description="Noise"),
    sim_time=widgets.FloatSlider(value=3,min=.5,max=10,step=.5,description="Time")
);

interactive(children=(FloatLogSlider(value=0.0005, description='Kp', max=-2.0, min=-5.0), FloatLogSlider(value…

## 5. Generic Controller → Actuator → Plant → Sensor — Fully Interactive

The user can now adjust:

- mechanical plant \(m,b,k\)
- controller \(K_p,K_i,K_d\)
- actuator gain and time constant
- sensor gain and time constant
- simulation time

The resulting closed-loop poles and stability are recalculated every time.

In [ ]:
def generic_loop_interactive(
    m=10.0,
    b=8.0,
    k=100.0,
    Kp=250.0,
    Ki=50.0,
    Kd=40.0,
    actuator_gain=5.0,
    actuator_tau=.1,
    sensor_gain=1.0,
    sensor_tau=.02,
    sim_time=10.0
):
    if not HAS_CONTROL:
        print("Install python-control first.")
        return

    G = ct.tf([1],[m,b,k])
    actuator = ct.tf([actuator_gain],[actuator_tau,1])
    controller = ct.tf([Kd,Kp,Ki],[1,0])
    sensor = ct.tf([sensor_gain],[sensor_tau,1])

    T = ct.feedback(controller*actuator*G, sensor)

    poles = ct.poles(T)
    response_type, stability, zeta = modal_label(poles)

    print("="*65)
    print("GENERIC CLOSED-LOOP ANALYSIS")
    print("="*65)
    print("Poles:", poles)
    print("Response:", response_type)
    print("Stability:", stability)
    if zeta is not None:
        print("Dominant damping ratio:", zeta)
    print("="*65)

    t=np.linspace(0,sim_time,1800)
    tt,y=ct.step_response(T,T=t)

    plt.figure(figsize=(10,4))
    plt.plot(tt,y)
    plt.axhline(1,linestyle="--")
    plt.grid(True)
    plt.title("Controller → Actuator → Plant → Sensor")
    plt.show()

widgets.interact(
    generic_loop_interactive,
    m=widgets.FloatSlider(value=10,min=1,max=50,step=1,description="Mass"),
    b=widgets.FloatSlider(value=8,min=0,max=100,step=1,description="Damping"),
    k=widgets.FloatSlider(value=100,min=10,max=500,step=10,description="Spring"),
    Kp=widgets.FloatSlider(value=250,min=0,max=1000,step=10,description="Kp"),
    Ki=widgets.FloatSlider(value=50,min=0,max=500,step=5,description="Ki"),
    Kd=widgets.FloatSlider(value=40,min=0,max=300,step=5,description="Kd"),
    actuator_gain=widgets.FloatSlider(value=5,min=.1,max=20,step=.1,description="Act gain"),
    actuator_tau=widgets.FloatSlider(value=.1,min=.01,max=2,step=.01,description="Act tau"),
    sensor_gain=widgets.FloatSlider(value=1,min=.1,max=5,step=.1,description="Sens gain"),
    sensor_tau=widgets.FloatSlider(value=.02,min=.005,max=.5,step=.005,description="Sens tau"),
    sim_time=widgets.FloatSlider(value=10,min=2,max=30,step=1,description="Time")
);

interactive(children=(FloatSlider(value=10.0, description='Mass', max=50.0, min=1.0, step=1.0), FloatSlider(va…

## 6. Cascade Control — Fully Interactive

The inner and outer loop gains can now be changed directly.

In [ ]:
def cascade_interactive(
    m=10,
    b=8,
    k=100,
    inner_gain=20,
    outer_Kp=2,
    outer_Ki=1,
    sim_time=10
):
    if not HAS_CONTROL:
        print("Install python-control first.")
        return

    G=ct.tf([1],[m,b,k])
    inner=ct.feedback(inner_gain*G,1)
    outer=ct.tf([outer_Kp,outer_Ki],[1,0])
    cascade=ct.feedback(outer*inner,1)

    poles=ct.poles(cascade)
    print("Poles:",poles)
    print("Stability:",stability_from_poles(poles))

    t=np.linspace(0,sim_time,1500)
    tt,y=ct.step_response(cascade,T=t)

    plt.figure(figsize=(10,4))
    plt.plot(tt,y)
    plt.axhline(1,linestyle="--")
    plt.grid(True)
    plt.title("Interactive Cascade Control")
    plt.show()

widgets.interact(
    cascade_interactive,
    m=widgets.FloatSlider(value=10,min=1,max=50,step=1,description="Mass"),
    b=widgets.FloatSlider(value=8,min=0,max=100,step=1,description="Damping"),
    k=widgets.FloatSlider(value=100,min=10,max=500,step=10,description="Spring"),
    inner_gain=widgets.FloatSlider(value=20,min=0,max=200,step=5,description="Inner K"),
    outer_Kp=widgets.FloatSlider(value=2,min=0,max=20,step=.5,description="Outer Kp"),
    outer_Ki=widgets.FloatSlider(value=1,min=0,max=10,step=.2,description="Outer Ki"),
    sim_time=widgets.FloatSlider(value=10,min=2,max=30,step=1,description="Time")
);

interactive(children=(FloatSlider(value=10.0, description='Mass', max=50.0, min=1.0, step=1.0), FloatSlider(va…

## 7. Feedforward + Feedback — Fully Interactive

You can now vary:

- plant parameters
- PI feedback controller
- feedforward gain
- simulation time

In [ ]:
def feedforward_interactive(
    m=10,
    b=8,
    k=100,
    Kp=2,
    Ki=1,
    feedforward_gain=100,
    sim_time=10
):
    if not HAS_CONTROL:
        print("Install python-control first.")
        return

    G=ct.tf([1],[m,b,k])
    C=ct.tf([Kp,Ki],[1,0])

    feedback=ct.feedback(C*G,1)
    feedforward_path=feedforward_gain*G/(1+C*G)
    combined=feedback+feedforward_path

    t=np.linspace(0,sim_time,1500)

    t1,y1=ct.step_response(feedback,T=t)
    t2,y2=ct.step_response(combined,T=t)

    plt.figure(figsize=(10,4))
    plt.plot(t1,y1,label="Feedback only")
    plt.plot(t2,y2,label="Feedforward + feedback")
    plt.axhline(1,linestyle="--")
    plt.grid(True)
    plt.legend()
    plt.title("Interactive Feedforward + Feedback")
    plt.show()

widgets.interact(
    feedforward_interactive,
    m=widgets.FloatSlider(value=10,min=1,max=50,step=1,description="Mass"),
    b=widgets.FloatSlider(value=8,min=0,max=100,step=1,description="Damping"),
    k=widgets.FloatSlider(value=100,min=10,max=500,step=10,description="Spring"),
    Kp=widgets.FloatSlider(value=2,min=0,max=20,step=.5,description="Kp"),
    Ki=widgets.FloatSlider(value=1,min=0,max=10,step=.2,description="Ki"),
    feedforward_gain=widgets.FloatSlider(value=100,min=0,max=500,step=10,description="FF gain"),
    sim_time=widgets.FloatSlider(value=10,min=2,max=30,step=1,description="Time")
);

interactive(children=(FloatSlider(value=10.0, description='Mass', max=50.0, min=1.0, step=1.0), FloatSlider(va…

## 8. Disturbance Rejection — Fully Interactive

This section lets you directly change the plant, controller gains, disturbance magnitude and simulation time.

In [ ]:
def disturbance_rejection_interactive(
    m=10,
    b=8,
    k=100,
    Kp=250,
    Ki=50,
    Kd=40,
    disturbance_magnitude=1.0,
    sim_time=10
):
    if not HAS_CONTROL:
        print("Install python-control first.")
        return

    G=ct.tf([1],[m,b,k])
    C=ct.tf([Kd,Kp,Ki],[1,0])
    L=C*G
    S=1/(1+L)
    Td=G*S*disturbance_magnitude

    t=np.linspace(0,sim_time,1500)
    tt,y=ct.step_response(Td,T=t)

    plt.figure(figsize=(10,4))
    plt.plot(tt,y)
    plt.grid(True)
    plt.title("Output Due to Input Disturbance")
    plt.xlabel("Time (s)")
    plt.show()

widgets.interact(
    disturbance_rejection_interactive,
    m=widgets.FloatSlider(value=10,min=1,max=50,step=1,description="Mass"),
    b=widgets.FloatSlider(value=8,min=0,max=100,step=1,description="Damping"),
    k=widgets.FloatSlider(value=100,min=10,max=500,step=10,description="Spring"),
    Kp=widgets.FloatSlider(value=250,min=0,max=1000,step=10,description="Kp"),
    Ki=widgets.FloatSlider(value=50,min=0,max=500,step=5,description="Ki"),
    Kd=widgets.FloatSlider(value=40,min=0,max=300,step=5,description="Kd"),
    disturbance_magnitude=widgets.FloatSlider(value=1,min=0,max=10,step=.5,description="Disturb."),
    sim_time=widgets.FloatSlider(value=10,min=2,max=30,step=1,description="Time")
);

interactive(children=(FloatSlider(value=10.0, description='Mass', max=50.0, min=1.0, step=1.0), FloatSlider(va…

## 9. Sensitivity and Complementary Sensitivity — Fully Interactive

The controller and plant parameters can be modified and the frequency-domain sensitivity curves are recalculated immediately.

In [ ]:
def sensitivity_interactive(
    m=10,
    b=8,
    k=100,
    Kp=250,
    Ki=50,
    Kd=40,
    w_min_exp=-2,
    w_max_exp=3
):
    if not HAS_CONTROL:
        print("Install python-control first.")
        return

    G=ct.tf([1],[m,b,k])
    C=ct.tf([Kd,Kp,Ki],[1,0])

    L=C*G
    S=1/(1+L)
    T=L/(1+L)

    w=np.logspace(w_min_exp,w_max_exp,1400)

    Sr=ct.frequency_response(S,w).frdata.squeeze()
    Tr=ct.frequency_response(T,w).frdata.squeeze()

    plt.figure(figsize=(10,5))
    plt.semilogx(w,20*np.log10(np.abs(Sr)),label="Sensitivity S")
    plt.semilogx(w,20*np.log10(np.abs(Tr)),label="Complementary Sensitivity T")
    plt.xlabel("Frequency (rad/s)")
    plt.ylabel("Magnitude (dB)")
    plt.grid(True)
    plt.legend()
    plt.title("Interactive Sensitivity Analysis")
    plt.show()

    print("Peak sensitivity Ms:",np.max(np.abs(Sr)))

widgets.interact(
    sensitivity_interactive,
    m=widgets.FloatSlider(value=10,min=1,max=50,step=1,description="Mass"),
    b=widgets.FloatSlider(value=8,min=0,max=100,step=1,description="Damping"),
    k=widgets.FloatSlider(value=100,min=10,max=500,step=10,description="Spring"),
    Kp=widgets.FloatSlider(value=250,min=0,max=1000,step=10,description="Kp"),
    Ki=widgets.FloatSlider(value=50,min=0,max=500,step=5,description="Ki"),
    Kd=widgets.FloatSlider(value=40,min=0,max=300,step=5,description="Kd"),
    w_min_exp=widgets.IntSlider(value=-2,min=-4,max=0,step=1,description="w min 10^"),
    w_max_exp=widgets.IntSlider(value=3,min=1,max=6,step=1,description="w max 10^")
);

interactive(children=(FloatSlider(value=10.0, description='Mass', max=50.0, min=1.0, step=1.0), FloatSlider(va…

## 10. Closed-Loop Stability Analysis — Fully Interactive

This section is dedicated to user-controlled stability analysis.

Change:

- plant mass, damping and stiffness
- controller gains

The notebook prints:

- poles
- response type
- stability
- dominant damping ratio

In [ ]:
def stability_interactive(
    m=10,
    b=8,
    k=100,
    Kp=250,
    Ki=50,
    Kd=40
):
    if not HAS_CONTROL:
        print("Install python-control first.")
        return

    G=ct.tf([1],[m,b,k])
    C=ct.tf([Kd,Kp,Ki],[1,0])
    closed_loop=ct.feedback(C*G,1)

    poles=ct.poles(closed_loop)
    response_type, stability, zeta=modal_label(poles)

    print("="*65)
    print("INTERACTIVE STABILITY ANALYSIS")
    print("="*65)
    print("Poles:",poles)
    print("Response type:",response_type)
    print("Stability:",stability)
    if zeta is not None:
        print("Dominant damping ratio:",zeta)
    print("="*65)

    plt.figure(figsize=(7,5))
    plt.scatter(np.real(poles),np.imag(poles),s=90)
    plt.axvline(0)
    plt.axhline(0)
    plt.grid(True)
    plt.xlabel("Real")
    plt.ylabel("Imaginary")
    plt.title("Closed-Loop Pole Map")
    plt.show()

widgets.interact(
    stability_interactive,
    m=widgets.FloatSlider(value=10,min=1,max=50,step=1,description="Mass"),
    b=widgets.FloatSlider(value=8,min=-50,max=100,step=1,description="Damping"),
    k=widgets.FloatSlider(value=100,min=10,max=500,step=10,description="Spring"),
    Kp=widgets.FloatSlider(value=250,min=0,max=1000,step=10,description="Kp"),
    Ki=widgets.FloatSlider(value=50,min=0,max=500,step=5,description="Ki"),
    Kd=widgets.FloatSlider(value=40,min=0,max=300,step=5,description="Kd")
);

interactive(children=(FloatSlider(value=10.0, description='Mass', max=50.0, min=1.0, step=1.0), FloatSlider(va…

# Summary

Every major simulation section in this notebook accepts user inputs.

You can directly change:

- physical plant parameters
- controller gains
- actuator dynamics
- sensor dynamics
- disturbances
- sensor noise
- saturation limits
- pneumatic pressure parameters
- hydraulic flow/compliance parameters
- cascade gains
- feedforward gain
- stability-analysis parameters
- simulation duration
- frequency-analysis range

The plots and printed stability classifications update automatically through `ipywidgets`.